# CME Futures: Latent-Factor Requests

The latent-factor stage contains two declared configurations. `10a_pca` fits principal components
within each training fold. `10b_stochastic_discount_factor` estimates the neural stochastic
discount factor within the same fold contract. Neither notebook selects by IC.

This index exposes the complete request population without launching either computation. The two
execution notebooks publish disjoint official populations that `13_backtest` later combines with
the other predictive families.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# CME 期貨：潛在因子請求

潛在因子階段包含兩項已宣告設定。`10a_pca` 在每個訓練折內擬合主成分。`10b_stochastic_discount_factor` 在相同資料折規約下估計神經網路隨機折現因子。兩份 Notebook 都不依資訊係數選擇。

此索引列出完整的訓練請求群組，但不啟動任何一項運算。兩份執行用 Notebook 會發布互不重疊的正式模型群組，之後由 `13_backtest` 將其與其他預測模型類別合併。
<!-- ML4T_ZH_END c0000-p000 -->


## What a latent factor is, and how this stage differs from the ones before it

Every model up to this point was handed named predictors. Carry, momentum, the volatility
estimate, the regime probability - each is a quantity somebody decided to compute, and the
model's job was to weigh them. The choice of what to compute came from the researcher, and a
driver nobody thought to name was a driver no model could use.

A latent factor is inferred instead of specified. The starting observation is that futures
returns move together far more than thirty independent series would: energy contracts rise and
fall as a group, the metals do, the equity indices do, and there are days on which nearly
everything moves the same way. That co-movement is evidence of a small number of underlying
drivers acting on many contracts at once. A latent-factor method estimates those drivers from
the covariance of returns themselves, without being told in advance what they are or how many
there should be.

The appeal is that it can find structure nobody encoded. The cost is that what it finds has no
name and no economic interpretation attached - a factor is a direction in return space that
explains variance, and whether it corresponds to anything a reader would recognise is a
separate question the method does not answer.

## Why two configurations, and what separates them

The two are not variations on one method. They disagree about what a factor is *for*, and
that disagreement is the reason both are here.


<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 什麼是潛在因子，以及本階段與先前階段有何不同

到目前為止，每個模型都取得具名的預測變數。持有收益、動能、波動率估計值、狀態機率，都是研究者決定計算的量，模型的工作是賦予它們權重。計算哪些量由研究者選擇；沒有想到要命名的驅動因素，模型便無從使用。

潛在因子則是推斷出來的，而非預先指定。起點是觀察到期貨報酬的共同變動，遠多於 30 條獨立序列應有的程度：能源合約成群漲跌、金屬如此、股價指數亦然，還有幾乎所有商品同向變動的日子。這種共同變動，顯示少數底層驅動因素同時作用於許多合約。潛在因子方法直接從報酬的共變異數估計這些因素，不必事先告知因素是什麼，或應有多少個。

它的吸引力在於能找出無人預先編碼的結構。代價是找到的因素沒有名稱，也沒有附帶經濟解釋：因子是在報酬空間中能解釋變異數的一個方向；它是否對應讀者熟悉的事物，則是此方法無法回答的另一個問題。

## 為何有兩項設定，以及其差異

兩者不是同一方法的不同版本。它們對因子的*用途*有不同看法，而這正是同時納入兩者的原因。
<!-- ML4T_ZH_END c0001-p000 -->

**`10a_pca` maximizes explained variance.** Principal components find the directions along
which returns vary most, in order, each uncorrelated with the ones before it. It is linear,
it has a closed-form solution, and it makes no reference to returns being predictable at all.
Its first component on a futures panel is typically close to "everything moves together"; the
next few usually separate the sectors. It is the standard baseline for exactly the reasons
equal weight is one in the backtest stage: it is well understood, it estimates little, and
anything more elaborate has to beat it to justify itself.

The weakness is that variance and return are different quantities. The direction along which
a panel varies most is not necessarily the direction that pays, and PCA has no mechanism for
preferring one that does - a factor capturing a large, entirely unrewarded common movement is
exactly what it is built to find first.

**`10b_stochastic_discount_factor` starts from what prices assets.** Asset pricing theory says
that if markets are free of arbitrage there exists a single random variable - the stochastic
discount factor - whose covariance with any asset's return explains that asset's expected
return. Everything that is priced is priced by the same object. The SDF is not observable, but
it is a well-defined thing to estimate, and estimating it with a neural network means not
having to assume in advance which functional form it takes.

The difference from PCA is the objective and the inputs, not the architecture. PCA asks which
directions explain the most variation; the SDF asks which combination best explains the
cross-section of *returns*. A factor that moves a lot but earns nothing is a success for the
first and a failure for the second.


<!-- ML4T_ZH_BEGIN c0001-p001 -->
**`10a_pca` 最大化可解釋變異數。**主成分分析會依序找出報酬變動最大的方向，每個方向與先前方向都不相關。它是線性方法，有封閉形式解，也完全不要求報酬具有可預測性。期貨面板的第一主成分通常接近「所有商品一起變動」；接下來幾個則通常區分不同產業類別。它是標準基準，理由與回測階段採用等權重基準相同：方法容易理解、估計參數少，更複雜的方法必須超越它，才能證明自身價值。

其弱點是變異數與報酬是不同的量。面板變動最大的方向不一定有報酬補償，而 PCA 也沒有偏好有報酬方向的機制。捕捉到幅度大、卻完全沒有報酬補償的共同變動因子，恰好就是它優先設計來尋找的對象。

**`10b_stochastic_discount_factor` 從資產如何定價出發。**資產定價理論指出，若市場不存在套利，就存在一個單一隨機變數，即隨機折現因子；它與任何資產報酬的共變異數，可解釋該資產的預期報酬。所有被定價的資產，都由同一對象定價。隨機折現因子無法直接觀察，但它是定義明確的估計對象；用神經網路估計時，無須預先假定其函數形式。

它與 PCA 的差異在於目標及輸入，而非架構。PCA 問哪些方向能解釋最多變動；隨機折現因子則問哪種組合最能解釋*報酬*的橫斷面。大幅變動卻沒有收益的因子，對前者是成功，對後者則是失敗。
<!-- ML4T_ZH_END c0001-p001 -->

They also see different data, which is easy to miss and changes what each can find.
`run_pca_fold` takes the characteristics panel and discards it with `del`, so PCA is handed
returns alone. `run_sdf_fold` passes the characteristics through, and the number of
instruments it builds is derived from their width. So the SDF can express "products with high
carry and low volatility load on this factor" and PCA structurally cannot, because PCA never
sees carry.

So the comparison between the two is not "which fits better". It is a question about this
panel: whether the directions along which futures returns vary most are also the directions
along which they are compensated. The two configurations are run under the same fold contract
and the same universe precisely so the comparison isolates that.

## Why the factors are fitted inside each fold, and why that matters more here

Both configurations estimate their factors within the training portion of each fold, never
once over the whole panel. That is the same discipline every other family follows, but the
consequence of breaking it is worse here and easier to miss.

A supervised model that saw future data would be caught by its own validation score looking
implausible. A latent-factor model fitted on the full sample fails more quietly: the factors
are estimated from the covariance of returns, so a factor fitted over 2011 to 2025 encodes
which contracts moved together across the entire period. Using it to form a position in 2014
means holding a portfolio constructed from the knowledge that those contracts would go on
co-moving. Nothing about the resulting prediction looks impossible. The returns are real, the
weights are finite, and the backtest runs - it just reports a strategy that could not have
been held.


<!-- ML4T_ZH_BEGIN c0001-p002 -->
兩者使用的資料也不同，這點容易忽略，卻會改變各自能找到的資訊。`run_pca_fold` 取得特徵面板，接著以 `del` 捨棄它，因此 PCA 只取得報酬。`run_sdf_fold` 則會傳入特徵，而建立的工具變數數量由特徵寬度決定。因此，隨機折現因子能表達「持有收益高、波動率低的商品對此因子有負荷」，PCA 在結構上做不到，因為它從未看到持有收益。

所以兩者的比較，並非「哪個擬合得更好」，而是在問這個面板中，期貨報酬變動最大的方向，是否也是獲得報酬補償的方向。兩項設定採用相同資料折規約及相同商品範圍，正是為了聚焦比較此點。

## 為何在每折內擬合因子，以及為何在此更重要

兩項設定都只在每折的訓練部分估計因子，絕不對整個面板一次估計。其他模型類別也遵循相同紀律，但在此違反規則的後果更嚴重，也更容易被忽略。

監督式模型若看過未來資料，其驗證分數異常地好，可能暴露問題。使用完整樣本擬合的潛在因子模型，則會更不顯眼地失效：因子由報酬共變異數估計，因此若以 2011 至 2025 年資料擬合，因子便編入了整段期間哪些合約共同變動的資訊。用它在 2014 年建立部位，等於持有一個以事先知道這些合約日後會共同變動為前提所建構的投資組合。產生的預測看起來沒有不可能之處；報酬是真實的，權重是有限值，回測也會執行，只是報告了一個當時不可能持有的策略。
<!-- ML4T_ZH_END c0001-p002 -->

The cost of doing it correctly is visible in what the early folds can support. A covariance
matrix over thirty products needs a meaningful amount of history before its estimate means
anything, so the earliest training window supports fewer reliable factors than the latest,
and a factor count fixed across folds is a compromise rather than a free choice. That is the
tradeoff the declared configuration is making, and it is the reason the count is declared in
`setup.yaml` rather than selected per fold - selecting it per fold on validation performance
would choose the number that best suited each window's outcomes.

## What the two tables below show

The universe table is the set of products the factors are estimated across. It is worth
reading before the request catalog, because a latent factor is a property of the panel rather
than of any one contract: adding or removing products changes what the factors are, in a way
that changing the universe for a per-product model does not. Two runs over different universes
do not produce comparable factors even under identical settings.

The request catalog is the complete declared population - one row per label and configuration,
resolved but unfitted. Reading it here is what makes the count the execution notebooks produce
checkable against a declaration.

## Why neither selects by IC, and why this page launches nothing

Both fit within each training fold, and both publish predictions like any other family. They
are not privileged by being unsupervised: their rows enter `13_backtest` alongside the linear,
gradient-boosting and sequence families and are selected on validation backtest Sharpe like
everything else. A high IC here decides nothing, which is the same rule the whole case study
runs under.


<!-- ML4T_ZH_BEGIN c0001-p003 -->
正確處理的代價，可從早期資料折所能支援的內容看出。涵蓋 30 項商品的共變異數矩陣，需要足夠長的歷史資料，估計值才有意義；因此最早的訓練視窗能可靠支援的因子數，少於較晚的視窗。在各折固定因子數是一項折衷，而非毫無代價的選擇。已宣告設定正是做出這項取捨，因此因子數寫在 `setup.yaml` 中，而非依每折選擇。若依各折的驗證表現選擇，就會挑出最符合該視窗結果的因子數。

## 下方兩張表顯示什麼

商品範圍表列出用來估計因子的商品集合。應先看這張表，再看請求目錄，因為潛在因子是整個面板的性質，而非單一合約的性質：增加或移除商品會改變因子本身，與改變逐商品模型的商品範圍不同。即使設定相同，商品範圍不同的兩次執行也不會產生可比較的因子。

請求目錄是已宣告的完整群組：每種標籤及設定各一列，已解析但尚未擬合。在此先閱讀它，才能將執行 Notebook 產出的數量，與事先宣告核對。

## 為何兩者都不依資訊係數選擇，以及此頁為何不啟動運算

兩者都在每個訓練折內擬合，也與其他模型類別一樣發布預測。無監督式特性並不賦予它們特殊待遇：其資料列會與線性、梯度提升及序列模型類別一起進入 `13_backtest`，並與其他設定一樣，依驗證回測夏普比率選擇。此處資訊係數偏高並不能決定任何事；整個案例研究都遵循相同規則。
<!-- ML4T_ZH_END c0001-p003 -->

This notebook computes neither. It exists so the declared request population can be read
before anything is fitted - the two execution notebooks publish disjoint official populations,
and seeing what they *will* contain is what makes a later count checkable against a
declaration rather than against whatever finished.

Disjoint is the part worth noticing. The two populations share no members, so `13_backtest`
combines rather than reconciles them, and a configuration missing from one is not covered by
the other being complete.

<!-- ML4T_ZH_BEGIN c0001-p004 -->
本 Notebook 不執行這兩種運算。它的用途是在任何擬合前，讓讀者看到已宣告的請求群組。兩份執行用 Notebook 會發布互不重疊的正式模型群組；先看到它們*將*包含什麼，日後才能對照宣告核查數量，而非只對照實際完成的內容。

值得注意的是兩個群組互不重疊。它們沒有共同成員，因此 `13_backtest` 是將兩者合併，而非調解彼此差異；其中一個群組若缺少某項設定，另一個群組完整也不能彌補。
<!-- ML4T_ZH_END c0001-p004 -->


In [1]:
"""Show the declared CME futures latent-factor requests."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    product_universe_table,
)

In [2]:
requests = model_request_catalog("latent_factors", labels=ALL_LABELS)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [3]:
requests.sort("label", "config_name")

family,label,config_name
str,str,str
"""latent_factors""","""fwd_ret_21d""","""pca"""
"""latent_factors""","""fwd_ret_21d""","""sdf"""
"""latent_factors""","""fwd_ret_5d""","""pca"""
"""latent_factors""","""fwd_ret_5d""","""sdf"""
